In [1]:
%cd ../..

/Users/hoangle/Projects/recsys


In [2]:
import sys
import pickle

import yaml
import polars as pl
from loguru import logger
from transformers import T5Tokenizer
from tqdm import tqdm

In [3]:
logger.remove()
logger.add(sys.stderr, level="DEBUG")

1

In [4]:
path = "src/gen_retrieval/configs.yaml"
with open(path) as file:
    conf = yaml.safe_load(file)

conf

{'PROJECT_NAME': 'DSI',
 'NUM_EPOCHS': 10,
 'BSZ': 20,
 'LR': '3e-6',
 'USE_LR_SCHEDULER': True,
 'MAX_LEN': 32,
 'INDEXING_RETRIEVAL_RATIO': 32,
 'LOGGER': 'tensorboard',
 'PATHS': {'ckpt_dir': 'ckpt/', 'logs': 'logs/'},
 'RAW_DATA': {'query': 'data/raw/hotpotqa/queries.jsonl',
  'corpus': 'data/raw/hotpotqa/corpus.jsonl',
  'train': 'data/raw/hotpotqa/qrels/train.tsv',
  'val': 'data/raw/hotpotqa/qrels/dev.tsv',
  'test': 'data/raw/hotpotqa/qrels/test.tsv'},
 'INTERIM': {'docid': {'tokens': 'data/interim/gen_retrieval/docid/corpus_tokens_[proc_id].npz',
   'embds': 'data/interim/gen_retrieval/docid/corpus_embds.pt'}},
 'PROCESSED': {'query': 'data/processed/gen_retrieval/query_tokenized.parquet',
  'corpus': 'data/processed/gen_retrieval/corpus_tokenized.parquet',
  'semantic_docid': 'data/processed/gen_retrieval/semantic_ids.pkl'},
 'MODEL_DOCID': 'bert-base-cased',
 'MODEL_GENERATIVE': 'res/t5-large',
 'NEW_TOKENS_DICT': {'task_indexing_tok': '<IDX>',
  'task_retrieval_tok': '<RTRV

# Define data

In [5]:
tokenizer = T5Tokenizer.from_pretrained(conf['MODEL_GENERATIVE'])
tokenizer.add_tokens(list(conf['NEW_TOKENS_DICT'].values()))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


3

# Process query

In [6]:
def _tokenize(text: str, tokenizer, max_length: int) -> tuple:
    out = tokenizer(text, add_special_tokens=True, max_length=max_length, truncation=True, padding="max_length")

    token_ids = '-'.join(list(map(str,out['input_ids'])))
    attn_mask = '-'.join(list(map(str,out['attention_mask'])))

    return token_ids, attn_mask

In [7]:
queries = pl.read_ndjson(conf['RAW_DATA']['query'])

In [8]:
token_ids_entries = []

for query in tqdm(queries.iter_rows(named=True), total=len(queries)):
    text = conf['NEW_TOKENS_DICT']['task_retrieval_tok'] + ' ' + query['text']
    token_ids_encoded, attn_masks_encoded = _tokenize(text, tokenizer, conf['MAX_LEN'])

    token_ids_entries.append({'_id': query['_id'], 'tok_ids': token_ids_encoded, 'attn_mask': attn_masks_encoded})

token_ids = pl.from_dicts(token_ids_entries)
token_ids.head()

100%|██████████| 97852/97852 [00:08<00:00, 12219.70it/s]


_id,tok_ids,attn_mask
str,str,str
"""5ab6d31155429954757d3384""","""32101-363-684-13-5233-405-1384…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5ac0d92f554299012d1db645""","""32101-571-186-18051-7-213-915-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5abd01335542993a06baf9fc""","""32101-4409-301-4667-35-63-6640…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5abff8c95542994516f4555c""","""32101-37-568-213-415-4387-845-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5adec8ad55429975fa854f8f""","""32101-37-7556-113-1944-7291-11…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"


In [9]:
queries_processed = (
    queries
    .drop('metadata')

    .join(token_ids, on='_id')
)

queries_processed.head()

_id,text,tok_ids,attn_mask
str,str,str,str
"""5ab6d31155429954757d3384""","""What country of origin does Ho…","""32101-363-684-13-5233-405-1384…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5ac0d92f554299012d1db645""","""How many fountains where prese…","""32101-571-186-18051-7-213-915-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5abd01335542993a06baf9fc""","""Chris Larceny directed the mus…","""32101-4409-301-4667-35-63-6640…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5abff8c95542994516f4555c""","""The person where local traditi…","""32101-37-568-213-415-4387-845-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
"""5adec8ad55429975fa854f8f""","""The actor who played Carl Swee…","""32101-37-7556-113-1944-7291-11…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"


## Save processed queries

In [10]:
path = conf['PROCESSED']['query']
queries_processed.write_parquet(path)

# Process corpus

In [11]:
corpus = (
    pl.read_ndjson(conf['RAW_DATA']['corpus'])
    .with_columns(
        pl.col('_id').cast(pl.UInt32),
    )
    .drop('title', 'metadata')
)

corpus.head()

_id,text
u32,str
12,"""Anarchism is a political philo…"
25,"""Autism is a neurodevelopmental…"
39,"""Albedo ( ) is a measure for re…"
290,"""A (named , plural ""As"", ""A's"",…"
303,"""Alabama ( ) is a state in the …"


## Tokenize corpus's text

In [12]:
token_ids_entries = []

for document in tqdm(corpus.iter_rows(named=True), total=len(corpus)):
    text = conf['NEW_TOKENS_DICT']['task_indexing_tok'] + ' ' + document["text"]
    token_ids_encoded, attn_masks_encoded = _tokenize(text, tokenizer, conf["MAX_LEN"])

    token_ids_entries.append(
        {"_id": document["_id"], "tok_ids_text": token_ids_encoded, "attn_mask_text": attn_masks_encoded}
    )

    # break

token_ids_text = pl.from_dicts(token_ids_entries)
token_ids_text.head()

100%|██████████| 5233329/5233329 [12:16<00:00, 7105.26it/s] 


_id,tok_ids_text,attn_mask_text
i64,str,str
12,"""32100-389-7064-159-51-19-3-9-1…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
25,"""32100-27777-19-3-9-6567-19677-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
39,"""32100-901-4143-32-41-3-61-19-3…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
290,"""32100-71-41-4350-26-3-6-28037-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"
303,"""32100-13050-41-3-61-19-3-9-538…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…"


## Tokenize corpus' semantic id

In [13]:
with open(conf["PROCESSED"]["semantic_docid"], "rb") as file:
    semantic_ids_raw = pickle.load(file)

In [14]:
tok_ids_semantic_ids_entries = []
for corpus_id, entry in tqdm(zip(corpus["_id"], semantic_ids_raw.values()), total=len(corpus)):
    text = " ".join(list(map(str, entry)))
    token_ids_encoded, attn_masks_encoded = _tokenize(text, tokenizer, conf["MAX_LEN"])

    tok_ids_semantic_ids_entries.append(
        {"_id": corpus_id, "tok_ids_semantic_id": token_ids_encoded, "attn_mask_semantic_id": attn_masks_encoded}
    )

    # break

token_ids_semantic_ids = pl.from_dicts(tok_ids_semantic_ids_entries)
token_ids_semantic_ids.head()

100%|██████████| 5233329/5233329 [03:59<00:00, 21827.36it/s]


_id,tok_ids_semantic_id,attn_mask_semantic_id
i64,str,str
12,"""505-505-220-489-305-314-204-20…","""1-1-1-1-1-1-1-1-1-1-1-0-0-0-0-…"
25,"""505-505-220-220-505-305-505-3-…","""1-1-1-1-1-1-1-1-1-1-0-0-0-0-0-…"
39,"""505-505-431-3-632-204-220-314-…","""1-1-1-1-1-1-1-1-1-1-1-0-0-0-0-…"
290,"""505-305-220-305-305-209-489-3-…","""1-1-1-1-1-1-1-1-1-1-0-0-0-0-0-…"
303,"""668-505-305-220-489-314-3-632-…","""1-1-1-1-1-1-1-1-1-0-0-0-0-0-0-…"


## Merge into single dataframe

In [15]:
corpus_processed = (
    corpus

    # Add token ids
    .join(token_ids_text, on='_id')

    # Add semantic ids
    .join(token_ids_semantic_ids, on='_id')
)

corpus_processed.head()

_id,text,tok_ids_text,attn_mask_text,tok_ids_semantic_id,attn_mask_semantic_id
u32,str,str,str,str,str
12,"""Anarchism is a political philo…","""32100-389-7064-159-51-19-3-9-1…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…","""505-505-220-489-305-314-204-20…","""1-1-1-1-1-1-1-1-1-1-1-0-0-0-0-…"
25,"""Autism is a neurodevelopmental…","""32100-27777-19-3-9-6567-19677-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…","""505-505-220-220-505-305-505-3-…","""1-1-1-1-1-1-1-1-1-1-0-0-0-0-0-…"
39,"""Albedo ( ) is a measure for re…","""32100-901-4143-32-41-3-61-19-3…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…","""505-505-431-3-632-204-220-314-…","""1-1-1-1-1-1-1-1-1-1-1-0-0-0-0-…"
290,"""A (named , plural ""As"", ""A's"",…","""32100-71-41-4350-26-3-6-28037-…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…","""505-305-220-305-305-209-489-3-…","""1-1-1-1-1-1-1-1-1-1-0-0-0-0-0-…"
303,"""Alabama ( ) is a state in the …","""32100-13050-41-3-61-19-3-9-538…","""1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-…","""668-505-305-220-489-314-3-632-…","""1-1-1-1-1-1-1-1-1-0-0-0-0-0-0-…"


## Save processed corpus

In [16]:
path = conf['PROCESSED']['corpus']
corpus_processed.write_parquet(path)